In [1]:
import os
import re
import sqlite3
import pandas as pd
import pickle
from typing import Tuple, List
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
VECTORSTORE_PATH = "./faiss_index"
SQLITE_PATH = "./argo_cleaned.db"
TABLE_NAME = "profiles"

In [3]:
embed = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.load_local(VECTORSTORE_PATH, embed, allow_dangerous_deserialization=True)

C:\Users\hitan\AppData\Local\Temp\ipykernel_18832\313234294.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embed = OpenAIEmbeddings(model="text-embedding-3-small")


In [6]:
from SCHEMA import DB_SCHEMA, PROMPT
db_schema = DB_SCHEMA.format(TABLE_NAME=TABLE_NAME)

In [9]:
def retrieve_context(query: str, k: int = 6):
    docs = vectorstore.similarity_search(query, k=k)
    contexts = [d.page_content for d in docs]
    return contexts, docs

In [ ]:
def get_sql_query(contexts: List[str], user_question: str) -> str:
    system_message = SystemMessage(content="You are an expert translator from English queries to SQL for the ARGO profiles database. You are given the database schema and some context information from the database. Use this information to generate accurate SQL queries.")
    human_template = PROMPT.format(table=TABLE_NAME, db_schema=db_schema, context="\n".join(contexts), question=user_question)
    human_message = HumanMessage(content=human_template)
    chatbot = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.0)
    result = chatbot.invoke([system_message, human_message])
    sql_text = result.content.strip()
    return sql_text
    

In [ ]:
def get_data_from_sql(sql_query):
    conn = sqlite3.connect(SQLITE_PATH)
    try:
        df = pd.read_sql_query(sql_query, conn)
    finally:
        conn.close()
    return df

In [ ]:
# "Show me salinity and temprature profiles from floats near 80 east and 90 east in january 2025"

In [ ]:
def main(user_question = None):
    if user_question:
        user_question = user_question
    else:
        user_question = input("Enter the Question: ")
    # user_question  = "Show me salinity and temprature profiles from floats near 80 east and 90 east in january 2025"
    contexts, docs = retrieve_context(user_question, k=10)
    sql_query = get_sql_query(contexts, user_question)
    print("Generated SQL Query: ",sql_query)
    df = get_data_from_sql(sql_query)
    return df
 

In [ ]:
df = main()
print(df.head())

Generated SQL Query:
SELECT PLATFORM_NUMBER, CYCLE_NUMBER, LATITUDE, LONGITUDE, PROFILE_DATE, PSAL, TEMP FROM profiles WHERE LONGITUDE BETWEEN 80 AND 90 AND PROFILE_DATE BETWEEN '2025-01-01' AND '2025-01-31' LIMIT 200


In [ ]:
conn = sqlite3.connect("argo_cleaned.db")

# Example: Get profiles from March 2025
df = pd.read_sql(f"{sql_query}", conn)

print(df)



    PLATFORM_NUMBER  CYCLE_NUMBER   LATITUDE  LONGITUDE PROFILE_DATE  \
0           2903988           1.0  13.450000  84.150000   2025-01-26   
1           2903988           1.0  13.450000  84.150000   2025-01-26   
2           2903988           1.0  13.450000  84.150000   2025-01-26   
3           2903988           1.0  13.450000  84.150000   2025-01-26   
4           2903988           1.0  13.450000  84.150000   2025-01-26   
..              ...           ...        ...        ...          ...   
195         7902248           1.0   9.916667  85.816667   2025-01-07   
196         7902248           1.0   9.916667  85.816667   2025-01-07   
197         7902248           1.0   9.916667  85.816667   2025-01-07   
198         7902248           1.0   9.916667  85.816667   2025-01-07   
199         7902248           1.0   9.916667  85.816667   2025-01-07   

          PSAL       TEMP  
0    33.529999  27.318001  
1    33.527000  27.323999  
2    33.526001  27.337000  
3    33.521000  27.3440